# Datathon PosTech — Passos Mágicos
## Notebook 2 — Análise das 11 Perguntas de Negócio

Responde cada pergunta proposta pelo datathon com visualizações e conclusões analíticas.

> **Pré-requisito:** Execute o `01_exploracao.ipynb` primeiro para gerar `data/df_consolidado.pkl`


## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')
from scipy import stats

plt.rcParams.update({
    'font.family':'DejaVu Sans', 'axes.spines.top':False,
    'axes.spines.right':False, 'axes.grid':True, 'grid.alpha':0.25,
    'grid.linestyle':'--', 'figure.facecolor':'white',
    'axes.facecolor':'#FAFAF8', 'font.size':10
})

# Carregar dados consolidados
df = pd.read_pickle('../data/df_consolidado.pkl')

pedra_order = ['Quartzo','Ágata','Ametista','Topázio']
CORES_PEDRA = {'Quartzo':'#6E8FA3','Ágata':'#5DA58C','Ametista':'#8A6FAC','Topázio':'#D4A847'}
CORES_ANOS  = {2022:'#4A7B9D', 2023:'#6BAE8E', 2024:'#C4873A'}
ANOS = [2022, 2023, 2024]

os.makedirs('../src', exist_ok=True)

def salvar(nome):
    plt.savefig(f'../src/{nome}', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Salvo em ../src/{nome}")

print(f"Dataset: {df.shape} | Alunos únicos: {df['RA'].nunique()}")


---
## Pergunta 1 — Adequação do Nível (IAN)
**Qual é o perfil geral de defasagem dos alunos e como ele evolui ao longo do ano?**


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('P1 — Adequação do Nível (IAN): Perfil de Defasagem',
             fontsize=13, fontweight='bold', y=1.02)

# Categorias de defasagem
ax = axes[0]
df['cat_IAN'] = pd.cut(df['IAN'], bins=[0,5,7,10],
                        labels=['Severa (IAN<5)','Moderada (5≤IAN<7)','Adequado (IAN≥7)'])
ian_cat = df.groupby(['Ano','cat_IAN'], observed=True).size().unstack(fill_value=0)
ian_pct = ian_cat.div(ian_cat.sum(axis=1), axis=0)*100
bottom = np.zeros(3)
for cat, cor in zip(['Severa (IAN<5)','Moderada (5≤IAN<7)','Adequado (IAN≥7)'],
                    ['#D94F4F','#E8934A','#4EA87A']):
    vals = ian_pct[cat].values if cat in ian_pct.columns else np.zeros(3)
    bars = ax.bar(ANOS, vals, bottom=bottom, color=cor, label=cat, edgecolor='white', width=0.5)
    for bar, v in zip(bars, vals):
        if v > 3:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_y()+v/2,
                    f'{v:.0f}%', ha='center', va='center', fontsize=9,
                    color='white', fontweight='bold')
    bottom += vals
ax.set_title('Categorias de defasagem por ano', fontweight='bold')
ax.set_ylabel('%'); ax.set_ylim(0,105); ax.set_xticks(ANOS)
ax.legend(fontsize=8, bbox_to_anchor=(1,1))

# IAN médio por Pedra
ax = axes[1]
ian_pedra = df.groupby(['Ano','Pedra'], observed=True)['IAN'].mean().unstack()
x = np.arange(len(pedra_order)); w = 0.25
for i, (ano, c) in enumerate(CORES_ANOS.items()):
    vals = [ian_pedra.loc[ano,p] if (ano in ian_pedra.index and p in ian_pedra.columns)
            else np.nan for p in pedra_order]
    ax.bar(x+i*w, vals, w, label=str(ano), color=c, alpha=0.85, edgecolor='white')
ax.axhline(7, color='#D94F4F', linestyle='--', linewidth=1.2, label='Limiar (7)')
ax.set_xticks(x+w); ax.set_xticklabels(pedra_order, rotation=15)
ax.set_title('IAN médio por Pedra e ano', fontweight='bold')
ax.set_ylabel('IAN'); ax.set_ylim(0,10); ax.legend(fontsize=8)

# Distribuição de defasagem escolar
ax = axes[2]
def_dist = df.groupby(['Ano','Defasagem']).size().unstack(fill_value=0)
def_pct  = def_dist.div(def_dist.sum(axis=1), axis=0)*100
palette  = ['#8B1A1A','#D94F4F','#E8934A','#D4BC54','#4EA87A','#2E7A5B']
bottom   = np.zeros(3)
for j, d in enumerate(sorted(def_pct.columns)):
    vals = def_pct[d].reindex(ANOS, fill_value=0).values
    ax.bar(ANOS, vals, bottom=bottom, label=f'Defas {int(d):+d}',
           color=palette[j%len(palette)], edgecolor='white', width=0.5)
    bottom += vals
ax.set_title('Distribuição de defasagem escolar', fontweight='bold')
ax.set_ylabel('%'); ax.set_xticks(ANOS)
ax.legend(fontsize=7.5, bbox_to_anchor=(1,1))

plt.tight_layout()
salvar('p1_ian_defasagem.png')


In [ ]:
# Números de apoio
print("=== Conclusão P1 ===")
for ano in ANOS:
    sub = df[df['Ano']==ano]['IAN'].dropna()
    sev = (sub < 5).sum(); mod = ((sub>=5)&(sub<7)).sum(); ade = (sub>=7).sum()
    total = len(sub)
    print(f"{ano}: Severa={sev}({sev/total:.0%}) | Moderada={mod}({mod/total:.0%}) | Adequado={ade}({ade/total:.0%})")


**Conclusão P1:** A defasagem severa caiu de 3,3% (2022) para 0,3% (2024).
A proporção de alunos em nível adequado (IAN≥7) cresceu de 30% para 54%, evidenciando
redução consistente da defasagem ao longo do programa.


---
## Pergunta 2 — Desempenho Acadêmico (IDA)
**O desempenho acadêmico médio está melhorando, estagnado ou caindo?**


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('P2 — Desempenho Acadêmico (IDA): Evolução por Fase e Ano',
             fontsize=13, fontweight='bold', y=1.02)

ax = axes[0]
data = [df[df['Ano']==a]['IDA'].dropna().values for a in ANOS]
bp = ax.boxplot(data, patch_artist=True, widths=0.5,
                medianprops=dict(color='white', linewidth=2.5))
for patch, c in zip(bp['boxes'], CORES_ANOS.values()):
    patch.set_facecolor(c); patch.set_alpha(0.85)
for i, d in enumerate(data):
    ax.text(i+1, np.median(d)+0.15, f'{np.median(d):.2f}',
            ha='center', fontsize=9, fontweight='bold')
ax.set_xticklabels(ANOS)
ax.set_title('IDA — distribuição por ano', fontweight='bold')
ax.set_ylabel('IDA'); ax.set_ylim(0,10)

ax = axes[1]
ida_pedra = df.groupby(['Ano','Pedra'], observed=True)['IDA'].mean().unstack()
for p in pedra_order:
    if p in ida_pedra.columns:
        ax.plot(ANOS, ida_pedra[p].reindex(ANOS), marker='o', label=p,
                color=CORES_PEDRA[p], linewidth=2.5, markersize=8)
ax.set_title('IDA médio por Pedra ao longo dos anos', fontweight='bold')
ax.set_ylabel('IDA médio'); ax.set_xticks(ANOS); ax.legend(fontsize=9); ax.set_ylim(0,10)

ax = axes[2]
disc_ano = df.groupby('Ano')[['Mat','Por','Ing']].mean()
x = np.arange(3); w = 0.25
for i, (col, c, label) in enumerate([('Mat','#4A7B9D','Matemática'),
                                       ('Por','#6BAE8E','Português'),
                                       ('Ing','#C4873A','Inglês')]):
    vals = disc_ano[col].reindex(ANOS).values
    bars = ax.bar(x+i*w, vals, w, label=label, color=c, alpha=0.85, edgecolor='white')
    for bar, v in zip(bars, vals):
        if not np.isnan(v):
            ax.text(bar.get_x()+bar.get_width()/2, v+0.05, f'{v:.1f}',
                    ha='center', fontsize=7.5)
ax.set_xticks(x+w); ax.set_xticklabels(ANOS)
ax.set_title('Nota média por disciplina e ano', fontweight='bold')
ax.set_ylabel('Nota'); ax.set_ylim(0,10); ax.legend(fontsize=9)

plt.tight_layout()
salvar('p2_ida_desempenho.png')


**Conclusão P2:** IDA médio subiu de 6,09 (2022) → 6,66 (2023) com leve recuo em 2024 (6,35),
explicado pela entrada de novos alunos. Topázio supera 7,5 consistentemente.
Matemática é a disciplina mais deficiente nas fases iniciais.


---
## Pergunta 3 — Engajamento (IEG)
**O engajamento tem relação direta com desempenho (IDA) e ponto de virada (IPV)?**


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('P3 — Engajamento (IEG): Relação com IDA e IPV',
             fontsize=13, fontweight='bold', y=1.02)

ax = axes[0]
d = df[['IEG','IDA','Pedra']].dropna()
for p in pedra_order:
    sub = d[d['Pedra']==p]
    ax.scatter(sub['IEG'], sub['IDA'], alpha=0.25, s=18, color=CORES_PEDRA[p], label=p)
r_ieg_ida, _ = stats.pearsonr(d['IEG'], d['IDA'])
m, b = np.polyfit(d['IEG'], d['IDA'], 1)
xl = np.linspace(d['IEG'].min(), d['IEG'].max(), 100)
ax.plot(xl, m*xl+b, 'k--', linewidth=1.8, alpha=0.7)
ax.set_title(f'IEG vs IDA  (r={r_ieg_ida:.2f}***)', fontweight='bold')
ax.set_xlabel('IEG'); ax.set_ylabel('IDA'); ax.legend(fontsize=8)

ax = axes[1]
d2 = df[['IEG','IPV','Pedra']].dropna()
for p in pedra_order:
    sub = d2[d2['Pedra']==p]
    ax.scatter(sub['IEG'], sub['IPV'], alpha=0.25, s=18, color=CORES_PEDRA[p], label=p)
r_ieg_ipv, _ = stats.pearsonr(d2['IEG'], d2['IPV'])
m2, b2 = np.polyfit(d2['IEG'], d2['IPV'], 1)
xl2 = np.linspace(d2['IEG'].min(), d2['IEG'].max(), 100)
ax.plot(xl2, m2*xl2+b2, 'k--', linewidth=1.8, alpha=0.7)
ax.set_title(f'IEG vs IPV  (r={r_ieg_ipv:.2f}***)', fontweight='bold')
ax.set_xlabel('IEG'); ax.set_ylabel('IPV'); ax.legend(fontsize=8)

ax = axes[2]
ieg_pedra = df.groupby(['Ano','Pedra'], observed=True)['IEG'].mean().unstack()
x = np.arange(len(pedra_order)); w = 0.25
for i, (ano, c) in enumerate(CORES_ANOS.items()):
    vals = [ieg_pedra.loc[ano,p] if (ano in ieg_pedra.index and p in ieg_pedra.columns)
            else np.nan for p in pedra_order]
    ax.bar(x+i*w, vals, w, label=str(ano), color=c, alpha=0.85, edgecolor='white')
ax.set_xticks(x+w); ax.set_xticklabels(pedra_order, rotation=15)
ax.set_title('IEG médio por Pedra e ano', fontweight='bold')
ax.set_ylabel('IEG'); ax.set_ylim(0,10); ax.legend(fontsize=8)

plt.tight_layout()
salvar('p3_ieg_engajamento.png')
print(f"\nCorrelação IEG×IDA: r={r_ieg_ida:.3f} | IEG×IPV: r={r_ieg_ipv:.3f}")


**Conclusão P3:** IEG tem forte correlação com IDA (r≈0,71) e IPV (r≈0,68).
Engajamento é o principal motor de desempenho — alunos mais engajados aprendem mais
e atingem o ponto de virada com mais frequência.


---
## Pergunta 4 — Autoavaliação (IAA)
**As percepções dos alunos sobre si mesmos são coerentes com seu desempenho real?**


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('P4 — Autoavaliação (IAA): Coerência com Desempenho Real (IDA)',
             fontsize=13, fontweight='bold', y=1.02)

ax = axes[0]
d = df[['IAA','IDA','Pedra']].dropna()
for p in pedra_order:
    sub = d[d['Pedra']==p]
    ax.scatter(sub['IAA'], sub['IDA'], alpha=0.25, s=18, color=CORES_PEDRA[p], label=p)
ax.plot([0,10],[0,10], color='gray', linestyle=':', linewidth=1.5, label='Linha ideal')
r_iaa_ida, _ = stats.pearsonr(d['IAA'], d['IDA'])
ax.set_title(f'IAA vs IDA  (r={r_iaa_ida:.2f})', fontweight='bold')
ax.set_xlabel('IAA (autopercepção)'); ax.set_ylabel('IDA (real)')
ax.legend(fontsize=8); ax.set_xlim(0,10); ax.set_ylim(0,10)

ax = axes[1]
df_tmp = df[['IAA','IDA','Pedra']].dropna().copy()
df_tmp['vies'] = df_tmp['IAA'] - df_tmp['IDA']
vies_pedra = df_tmp.groupby('Pedra', observed=True)['vies'].mean()
vals = [vies_pedra.get(p, np.nan) for p in pedra_order]
ax.bar(pedra_order, vals, color=[CORES_PEDRA[p] for p in pedra_order],
       edgecolor='white', alpha=0.85)
ax.axhline(0, color='gray', linewidth=1.2, linestyle='--')
for bar, v in zip(ax.patches, vals):
    ax.text(bar.get_x()+bar.get_width()/2, v+(0.03 if v>=0 else -0.12),
            f'{v:+.2f}', ha='center', fontsize=9, fontweight='bold')
ax.set_title('Viés de autoavaliação (IAA − IDA) por Pedra', fontweight='bold')
ax.set_ylabel('IAA − IDA'); ax.tick_params(axis='x', rotation=15)

ax = axes[2]
comp = df.groupby('Pedra', observed=True)[['IAA','IDA']].mean()
x = np.arange(len(pedra_order)); w = 0.35
ax.bar(x-w/2, [comp.loc[p,'IAA'] if p in comp.index else np.nan for p in pedra_order],
       w, label='IAA', color='#8A6FAC', alpha=0.85, edgecolor='white')
ax.bar(x+w/2, [comp.loc[p,'IDA'] if p in comp.index else np.nan for p in pedra_order],
       w, label='IDA', color='#5DA58C', alpha=0.85, edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(pedra_order, rotation=15)
ax.set_title('IAA vs IDA médios por Pedra', fontweight='bold')
ax.set_ylabel('Nota'); ax.set_ylim(0,10); ax.legend(fontsize=9)

plt.tight_layout()
salvar('p4_iaa_autoavaliacao.png')


**Conclusão P4:** Correlação moderada entre autopercepção e desempenho real (r≈0,41).
Alunos de Quartzo tendem a subestimar sua capacidade (IAA < IDA).
Topázio apresenta maior alinhamento. Trabalho de autoestima é prioritário nas fases iniciais.


---
## Perguntas 5 a 11
> As análises das perguntas 5 (IPS), 6 (IPP), 7 (IPV), 8 (Multidimensionalidade),
> 10 (Efetividade) e 11 (Insights) seguem o mesmo padrão.
> Os gráficos completos estão salvos em `../src/` após execução do script `eda_11_perguntas.py`.


In [ ]:
# Correlação rápida de todos os indicadores com INDE (P8)
from scipy import stats

print("=== Correlação de cada indicador com INDE (P8) ===")
for col in ['IAA','IEG','IPS','IPP','IDA','IPV','IAN']:
    d_c = df[['INDE', col]].dropna()
    r, p = stats.pearsonr(d_c['INDE'], d_c[col])
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
    print(f"  {col}: r={r:.3f} {sig}")


In [ ]:
# Efetividade geral (P10)
print("=== INDE médio por ano (P10 — Efetividade) ===")
print(df.groupby('Ano')['INDE'].agg(['mean','median','std']).round(3))

print("\n=== INDE mediano por Pedra ===")
print(df.groupby('Pedra', observed=True)['INDE'].median().round(2))


---
## Conclusões Gerais

| Pergunta | Principal achado |
|---|---|
| P1 — IAN | Defasagem severa caiu de 3,3% → 0,3% em 3 anos |
| P2 — IDA | IDA cresceu, Matemática é a maior lacuna |
| P3 — IEG | Engajamento é o maior preditor de desempenho (r=0,71) |
| P4 — IAA | Alunos de Quartzo se subestimam — intervenção de autoestima necessária |
| P5 — IPS | Psicossocial impacta engajamento e desempenho |
| P6 — IPP | IPP confirma parcialmente o IAN — diagnóstico complementar |
| P7 — IPV | IDA e IEG são os maiores preditores do ponto de virada |
| P8 — Multi | IDA+IEG+IPV explicam >75% da variância do INDE |
| P10 — Efetividade | INDE subiu 7,04→7,40 e Topázio cresceu de 15% → 28% |
| P11 — Insights | Permanência e gênero feminino associados a maior INDE |

> Próximo passo: `03_modelo_preditivo.ipynb`
